# App Review Source Technical Validation - Run 3

This notebook is a clean Run 3 validation for public app review collection sources.

The goal is to test whether Google Play and iOS App Store public review sources can support a recurring ingestion workflow.

The validation focuses on:

- access method
- review volume
- available metadata fields
- pagination and batching
- duplicate handling
- freshness
- field consistency
- request failures
- exploratory data analysis
- data quality issues

Google Play is tested as the primary source. iOS App Store public RSS feed is tested as a secondary source.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from datetime import datetime

PROJECT_DIR = "/content/drive/MyDrive/app_review_source_validation"

folders = [
    "data/raw",
    "data/processed",
    "outputs/summaries",
    "outputs/figures",
    "reports"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

RUN_DATE = datetime.now().strftime("%Y_%m_%d")
RUN_LABEL = "run3"

print("Project folder:", PROJECT_DIR)
print("Run date:", RUN_DATE)
print("Run label:", RUN_LABEL)

Project folder: /content/drive/MyDrive/app_review_source_validation
Run date: 2026_06_23
Run label: run3


In [3]:
!pip install google-play-scraper pandas requests matplotlib langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=d7992f88e8c6a640b8f92ff7355fb2b7101f1090ff11f388056528e50466c311
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [4]:
import pandas as pd
import requests
import time
import matplotlib.pyplot as plt

from datetime import datetime
from google_play_scraper import reviews, Sort
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

In [5]:
apps = [
    {
        "name": "YouTube",
        "google_package": "com.google.android.youtube",
        "ios_id": "544007664"
    },
    {
        "name": "TikTok",
        "google_package": "com.zhiliaoapp.musically",
        "ios_id": "835599320"
    },
    {
        "name": "Spotify",
        "google_package": "com.spotify.music",
        "ios_id": "324684580"
    }
]

countries = ["us", "gb", "ca", "au", "in"]

apps

[{'name': 'YouTube',
  'google_package': 'com.google.android.youtube',
  'ios_id': '544007664'},
 {'name': 'TikTok',
  'google_package': 'com.zhiliaoapp.musically',
  'ios_id': '835599320'},
 {'name': 'Spotify',
  'google_package': 'com.spotify.music',
  'ios_id': '324684580'}]

## 1. Google Play Review Collection

Google Play is tested as the primary source using the `google-play-scraper` Python package.

This path does not require Google Play Console owner access. For Run 2, I collected the newest US / English reviews for each app.

The test checks:

- whether public reviews can be collected
- whether batching works
- whether review volume is sufficient
- whether review IDs are available for duplicate handling
- whether the newest reviews are fresh
- whether metadata fields are usable for downstream analysis

In [6]:
def collect_google_play(app_list, target_count=2000, country="us", lang="en"):
    rows = []
    summary = []

    for app in app_list:
        app_name = app["name"]
        package = app["google_package"]

        token = None
        total = 0
        failed = False
        error_message = None
        start = time.time()

        while total < target_count:
            try:
                batch, token = reviews(
                    package,
                    lang=lang,
                    country=country,
                    sort=Sort.NEWEST,
                    count=min(200, target_count - total),
                    continuation_token=token
                )

                if len(batch) == 0:
                    break

                for r in batch:
                    rows.append({
                        "source": "google_play",
                        "app_name": app_name,
                        "app_package": package,
                        "country": country,
                        "language": lang,
                        "review_id": r.get("reviewId"),
                        "user_name": r.get("userName"),
                        "review_text": r.get("content"),
                        "rating": r.get("score"),
                        "thumbs_up": r.get("thumbsUpCount"),
                        "review_created_version": r.get("reviewCreatedVersion"),
                        "review_date": r.get("at"),
                        "developer_reply": r.get("replyContent"),
                        "reply_date": r.get("repliedAt"),
                        "app_version": r.get("appVersion")
                    })

                total += len(batch)

                if token is None:
                    break

                time.sleep(0.3)

            except Exception as e:
                failed = True
                error_message = str(e)
                break

        runtime_seconds = round(time.time() - start, 2)

        summary.append({
            "source": "google_play",
            "app_name": app_name,
            "app_package": package,
            "country": country,
            "language": lang,
            "target_count": target_count,
            "collected_count": total,
            "failed": failed,
            "error_message": error_message,
            "runtime_seconds": runtime_seconds
        })

    return pd.DataFrame(rows), pd.DataFrame(summary)

In [7]:
google_reviews, google_collection_summary = collect_google_play(apps, target_count=2000)

google_raw_path = os.path.join(PROJECT_DIR, f"data/raw/google_play_reviews_{RUN_LABEL}_{RUN_DATE}.csv")
google_summary_path = os.path.join(PROJECT_DIR, f"outputs/summaries/google_play_collection_summary_{RUN_LABEL}_{RUN_DATE}.csv")

google_reviews.to_csv(google_raw_path, index=False)
google_collection_summary.to_csv(google_summary_path, index=False)

display(google_collection_summary)

print("Saved raw data to:", google_raw_path)
print("Saved summary to:", google_summary_path)
print("Total Google Play reviews collected:", len(google_reviews))

,source,app_name,app_package,country,language,target_count,collected_count,failed,error_message,runtime_seconds
0,google_play,YouTube,com.google.android.youtube,us,en,2000,2000,False,None,5.98
1,google_play,TikTok,com.zhiliaoapp.musically,us,en,2000,2000,False,None,5.38
2,google_play,Spotify,com.spotify.music,us,en,2000,2000,False,None,5.62


Saved raw data to: /content/drive/MyDrive/app_review_source_validation/data/raw/google_play_reviews_run3_2026_06_23.csv
Saved summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_collection_summary_run3_2026_06_23.csv
Total Google Play reviews collected: 6000


## 2. Google Play Validation Summary

This section summarizes the Google Play Run 2 result.

The goal is to check review volume, review ID uniqueness, freshness, duplicate count, and metadata coverage for each app.

In [8]:
google_validation_summary = (
    google_reviews
    .groupby("app_name")
    .agg(
        review_count=("review_id", "count"),
        unique_review_ids=("review_id", "nunique"),
        oldest_review=("review_date", "min"),
        newest_review=("review_date", "max"),
        avg_rating=("rating", "mean"),
        reviews_with_text=("review_text", lambda x: x.notna().sum()),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum()),
        reviews_with_review_created_version=("review_created_version", lambda x: x.notna().sum()),
        reviews_with_developer_reply=("developer_reply", lambda x: x.notna().sum())
    )
    .reset_index()
)

google_validation_summary["duplicate_count"] = (
    google_validation_summary["review_count"] - google_validation_summary["unique_review_ids"]
)

google_validation_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_validation_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

google_validation_summary.to_csv(google_validation_path, index=False)

display(google_validation_summary)

print("Saved validation summary to:", google_validation_path)

,app_name,review_count,unique_review_ids,oldest_review,newest_review,avg_rating,reviews_with_text,reviews_with_app_version,reviews_with_review_created_version,reviews_with_developer_reply,duplicate_count
0,Spotify,2000,2000,2026-06-20 09:14:01,2026-06-22 18:36:20,3.5830,2000,1656,1656,317,0
1,TikTok,2000,2000,2026-06-19 23:47:35,2026-06-22 18:37:36,3.8985,2000,1293,1293,1659,0
2,YouTube,2000,2000,2026-06-21 12:35:02,2026-06-22 18:36:47,3.5585,2000,1957,1957,0,0


Saved validation summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_validation_summary_run3_2026_06_23.csv


## 3. Duplicate Handling

For repeated ingestion, each review needs a stable review key.

For Google Play, I use source, app name, country, language, and review ID.

For iOS, I use source, app name, country, and review ID.

This allows future runs to identify whether a review is new or already collected.

In [9]:
def add_review_key(df):
    df = df.copy()

    for col in ["source", "app_name", "country", "review_id"]:
        if col not in df.columns:
            df[col] = ""

    if "language" not in df.columns:
        df["language"] = ""

    df["review_key"] = (
        df["source"].astype(str) + "_" +
        df["app_name"].astype(str) + "_" +
        df["country"].astype(str) + "_" +
        df["language"].astype(str) + "_" +
        df["review_id"].astype(str)
    )

    return df


google_reviews_checked = add_review_key(google_reviews)

google_duplicate_summary = pd.DataFrame([
    {
        "source": "google_play",
        "total_reviews": len(google_reviews_checked),
        "unique_review_keys": google_reviews_checked["review_key"].nunique(),
        "duplicate_rows": google_reviews_checked.duplicated("review_key").sum()
    }
])

google_duplicate_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_duplicate_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

google_duplicate_summary.to_csv(google_duplicate_path, index=False)

display(google_duplicate_summary)

print("Saved duplicate summary to:", google_duplicate_path)

,source,total_reviews,unique_review_keys,duplicate_rows
0,google_play,6000,6000,0


Saved duplicate summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_duplicate_summary_run3_2026_06_23.csv


## 4. iOS App Store Public Review Feed

iOS App Store reviews are tested using the public RSS review feed.

This path does not require App Store Connect owner access. However, the public feed has stronger limitations than Google Play, so this source is tested as a secondary path.

For Run 2, I tested multiple countries and pages to check whether country rotation and pagination can improve review coverage.

In [10]:
def read_nested_value(obj, keys, default=None):
    value = obj

    for key in keys:
        if isinstance(value, dict):
            value = value.get(key)
        else:
            return default

    if value is None:
        return default

    return value


def collect_ios_reviews(app_list, country_list, max_pages=10):
    rows = []
    summary = []

    for app in app_list:
        app_name = app["name"]
        ios_id = app["ios_id"]

        for country in country_list:
            total = 0
            empty_pages = 0
            failed_pages = 0
            start = time.time()

            for page in range(1, max_pages + 1):
                url = (
                    f"https://itunes.apple.com/{country}/rss/customerreviews/"
                    f"page={page}/id={ios_id}/sortby=mostrecent/json"
                )

                try:
                    response = requests.get(url, timeout=20)

                    if response.status_code != 200:
                        failed_pages += 1
                        continue

                    data = response.json()
                    entries = data.get("feed", {}).get("entry", [])

                    if isinstance(entries, dict):
                        entries = [entries]

                    review_entries = [
                        e for e in entries
                        if "im:rating" in e and "content" in e
                    ]

                    if len(review_entries) == 0:
                        empty_pages += 1
                        continue

                    for r in review_entries:
                        rows.append({
                            "source": "ios_app_store",
                            "app_name": app_name,
                            "ios_id": ios_id,
                            "country": country,
                            "page": page,
                            "review_id": read_nested_value(r, ["id", "label"]),
                            "user_name": read_nested_value(r, ["author", "name", "label"]),
                            "rating": read_nested_value(r, ["im:rating", "label"]),
                            "title": read_nested_value(r, ["title", "label"]),
                            "review_text": read_nested_value(r, ["content", "label"]),
                            "app_version": read_nested_value(r, ["im:version", "label"]),
                            "review_date": read_nested_value(r, ["updated", "label"]),
                            "review_url": read_nested_value(r, ["link", "attributes", "href"])
                        })

                    total += len(review_entries)
                    time.sleep(0.3)

                except Exception:
                    failed_pages += 1
                    continue

            runtime_seconds = round(time.time() - start, 2)

            summary.append({
                "source": "ios_app_store",
                "app_name": app_name,
                "ios_id": ios_id,
                "country": country,
                "pages_requested": max_pages,
                "collected_count": total,
                "empty_pages": empty_pages,
                "failed_pages": failed_pages,
                "runtime_seconds": runtime_seconds
            })

    return pd.DataFrame(rows), pd.DataFrame(summary)

In [11]:
ios_reviews, ios_collection_summary = collect_ios_reviews(apps, countries, max_pages=10)

ios_raw_path = os.path.join(
    PROJECT_DIR,
    f"data/raw/ios_reviews_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_collection_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_reviews.to_csv(ios_raw_path, index=False)
ios_collection_summary.to_csv(ios_summary_path, index=False)

display(ios_collection_summary)

print("Saved iOS raw data to:", ios_raw_path)
print("Saved iOS summary to:", ios_summary_path)
print("Total iOS reviews collected:", len(ios_reviews))

,source,app_name,ios_id,country,pages_requested,collected_count,empty_pages,failed_pages,runtime_seconds
0,ios_app_store,YouTube,544007664,us,10,500,0,0,6.89
1,ios_app_store,YouTube,544007664,gb,10,500,0,0,6.19
2,ios_app_store,YouTube,544007664,ca,10,500,0,0,5.26
3,ios_app_store,YouTube,544007664,au,10,500,0,0,6.23
4,ios_app_store,YouTube,544007664,in,10,500,0,0,7.21
5,ios_app_store,TikTok,835599320,us,10,450,1,0,8.09
6,ios_app_store,TikTok,835599320,gb,10,450,1,0,5.07
7,ios_app_store,TikTok,835599320,ca,10,500,0,0,5.24
8,ios_app_store,TikTok,835599320,au,10,500,0,0,5.13
9,ios_app_store,TikTok,835599320,in,10,0,10,0,1.51


Saved iOS raw data to: /content/drive/MyDrive/app_review_source_validation/data/raw/ios_reviews_run3_2026_06_23.csv
Saved iOS summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_collection_summary_run3_2026_06_23.csv
Total iOS reviews collected: 6800


## 5. iOS Validation Summary

This section summarizes the iOS App Store public RSS Run 2 result.

The goal is to check country/page coverage, review volume, empty pages, failed pages, and basic metadata availability.

Compared with Google Play, iOS requires country and page rotation to increase review coverage.

In [12]:
ios_validation_summary = (
    ios_reviews
    .groupby(["app_name", "country"])
    .agg(
        review_count=("review_id", "count"),
        unique_review_ids=("review_id", "nunique"),
        oldest_review=("review_date", "min"),
        newest_review=("review_date", "max"),
        reviews_with_text=("review_text", lambda x: x.notna().sum()),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum())
    )
    .reset_index()
)

ios_validation_summary["duplicate_count"] = (
    ios_validation_summary["review_count"] - ios_validation_summary["unique_review_ids"]
)

ios_validation_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_validation_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_validation_summary.to_csv(ios_validation_path, index=False)

display(ios_validation_summary)

print("Saved iOS validation summary to:", ios_validation_path)

,app_name,country,review_count,unique_review_ids,oldest_review,newest_review,reviews_with_text,reviews_with_app_version,duplicate_count
0,Spotify,au,500,500,2026-06-06T16:41:59-07:00,2026-06-22T04:39:20-07:00,500,500,0
1,Spotify,ca,450,450,2026-06-10T13:01:47-07:00,2026-06-20T15:09:11-07:00,450,450,0
2,Spotify,gb,450,450,2026-06-11T15:32:41-07:00,2026-06-22T02:14:41-07:00,450,450,0
3,Spotify,in,500,500,2026-06-06T09:29:58-07:00,2026-06-22T03:36:06-07:00,500,500,0
4,Spotify,us,500,500,2026-06-20T16:21:24-07:00,2026-06-22T04:39:51-07:00,500,500,0
5,TikTok,au,500,500,2026-02-26T13:13:33-07:00,2026-06-22T02:08:41-07:00,500,500,0
6,TikTok,ca,500,450,2026-04-18T21:43:06-07:00,2026-06-21T22:00:04-07:00,500,500,50
7,TikTok,gb,450,450,2026-05-30T00:26:47-07:00,2026-06-22T03:42:50-07:00,450,450,0
8,TikTok,us,450,450,2026-06-17T23:33:45-07:00,2026-06-22T04:18:31-07:00,450,450,0
9,YouTube,au,500,500,2026-05-02T01:35:00-07:00,2026-06-22T04:11:01-07:00,500,500,0


Saved iOS validation summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_validation_summary_run3_2026_06_23.csv


In [13]:
ios_reviews_checked = add_review_key(ios_reviews)

ios_duplicate_summary = pd.DataFrame([
    {
        "source": "ios_app_store",
        "total_reviews": len(ios_reviews_checked),
        "unique_review_keys": ios_reviews_checked["review_key"].nunique(),
        "duplicate_rows": ios_reviews_checked.duplicated("review_key").sum()
    }
])

ios_duplicate_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_duplicate_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_duplicate_summary.to_csv(ios_duplicate_path, index=False)

display(ios_duplicate_summary)

print("Saved iOS duplicate summary to:", ios_duplicate_path)

,source,total_reviews,unique_review_keys,duplicate_rows
0,ios_app_store,6800,6750,50


Saved iOS duplicate summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_duplicate_summary_run3_2026_06_23.csv


## 6. Source-Level Comparison

This section compares Google Play and iOS App Store public review sources at a high level.

The comparison focuses on collected volume, duplicate rate, failed requests, and coverage limitations.

In [14]:
source_comparison = pd.DataFrame([
    {
        "source": "google_play",
        "total_reviews": len(google_reviews_checked),
        "unique_review_keys": google_reviews_checked["review_key"].nunique(),
        "duplicate_rows": google_reviews_checked.duplicated("review_key").sum(),
        "failed_requests_or_apps": google_collection_summary["failed"].sum(),
        "main_limitation": "Needs repeated runs to confirm freshness and new review capture"
    },
    {
        "source": "ios_app_store",
        "total_reviews": len(ios_reviews_checked),
        "unique_review_keys": ios_reviews_checked["review_key"].nunique(),
        "duplicate_rows": ios_reviews_checked.duplicated("review_key").sum(),
        "failed_requests_or_apps": ios_collection_summary["failed_pages"].sum(),
        "main_limitation": "Country/page coverage can be inconsistent; some app-country pairs may return empty pages"
    }
])

source_comparison["duplicate_rate"] = (
    source_comparison["duplicate_rows"] / source_comparison["total_reviews"]
)

source_comparison_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/source_comparison_{RUN_LABEL}_{RUN_DATE}.csv"
)

source_comparison.to_csv(source_comparison_path, index=False)

display(source_comparison)

print("Saved source comparison to:", source_comparison_path)

,source,total_reviews,unique_review_keys,duplicate_rows,failed_requests_or_apps,main_limitation,duplicate_rate
0,google_play,6000,6000,0,0,Needs repeated runs to confirm freshness and n...,0.000000
1,ios_app_store,6800,6750,50,0,Country/page coverage can be inconsistent; som...,0.007353


Saved source comparison to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/source_comparison_run3_2026_06_23.csv


## 7. Exploratory and Data Quality Analysis

This section looks at the collected review data itself, not only whether the collection script works.

The checks include:

- rating distribution
- review length
- missing fields
- app version coverage
- detected language
- potential data quality issues

Google Play is still the main focus, but I also include iOS where the field structure is comparable.

In [15]:
google_rating_distribution = (
    google_reviews_checked
    .groupby(["source", "app_name", "rating"])
    .size()
    .reset_index(name="review_count")
)

ios_reviews_checked["rating"] = pd.to_numeric(ios_reviews_checked["rating"], errors="coerce")

ios_rating_distribution = (
    ios_reviews_checked
    .groupby(["source", "app_name", "rating"])
    .size()
    .reset_index(name="review_count")
)

rating_distribution = pd.concat(
    [google_rating_distribution, ios_rating_distribution],
    ignore_index=True
)

rating_distribution_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/rating_distribution_{RUN_LABEL}_{RUN_DATE}.csv"
)

rating_distribution.to_csv(rating_distribution_path, index=False)

display(rating_distribution)

print("Saved rating distribution to:", rating_distribution_path)

,source,app_name,rating,review_count
0,google_play,Spotify,1,498
1,google_play,Spotify,2,132
2,google_play,Spotify,3,124
3,google_play,Spotify,4,198
4,google_play,Spotify,5,1048
5,google_play,TikTok,1,382
6,google_play,TikTok,2,111
7,google_play,TikTok,3,108
8,google_play,TikTok,4,126
9,google_play,TikTok,5,1273


Saved rating distribution to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/rating_distribution_run3_2026_06_23.csv


In [16]:
def add_review_length(df):
    df = df.copy()
    df["review_text"] = df["review_text"].fillna("")
    df["review_char_length"] = df["review_text"].str.len()
    df["review_word_length"] = df["review_text"].str.split().str.len()
    return df


google_reviews_checked = add_review_length(google_reviews_checked)
ios_reviews_checked = add_review_length(ios_reviews_checked)

google_review_length_summary = (
    google_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        avg_char_length=("review_char_length", "mean"),
        median_char_length=("review_char_length", "median"),
        avg_word_length=("review_word_length", "mean"),
        median_word_length=("review_word_length", "median")
    )
    .reset_index()
)

ios_review_length_summary = (
    ios_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        avg_char_length=("review_char_length", "mean"),
        median_char_length=("review_char_length", "median"),
        avg_word_length=("review_word_length", "mean"),
        median_word_length=("review_word_length", "median")
    )
    .reset_index()
)

review_length_summary = pd.concat(
    [google_review_length_summary, ios_review_length_summary],
    ignore_index=True
)

review_length_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/review_length_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

review_length_summary.to_csv(review_length_path, index=False)

display(review_length_summary)

print("Saved review length summary to:", review_length_path)

,source,app_name,avg_char_length,median_char_length,avg_word_length,median_word_length
0,google_play,Spotify,86.754000,39.0,16.531000,8.0
1,google_play,TikTok,57.225500,19.0,10.952000,4.0
2,google_play,YouTube,58.708500,16.0,10.978500,3.0
3,ios_app_store,Spotify,93.295833,50.0,17.950000,10.0
4,ios_app_store,TikTok,178.940526,124.0,34.111579,24.0
5,ios_app_store,YouTube,137.937600,71.0,25.332400,14.0


Saved review length summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/review_length_summary_run3_2026_06_23.csv


In [17]:
google_fields_to_check = [
    "review_id",
    "user_name",
    "review_text",
    "rating",
    "thumbs_up",
    "review_created_version",
    "review_date",
    "developer_reply",
    "reply_date",
    "app_version"
]

google_missing_summary = (
    google_reviews_checked[google_fields_to_check]
    .isna()
    .mean()
    .reset_index()
)

google_missing_summary.columns = ["field", "missing_rate"]
google_missing_summary["source"] = "google_play"


ios_fields_to_check = [
    "review_id",
    "user_name",
    "review_text",
    "rating",
    "title",
    "app_version",
    "review_date",
    "review_url"
]

ios_missing_summary = (
    ios_reviews_checked[ios_fields_to_check]
    .isna()
    .mean()
    .reset_index()
)

ios_missing_summary.columns = ["field", "missing_rate"]
ios_missing_summary["source"] = "ios_app_store"


missing_summary = pd.concat(
    [google_missing_summary, ios_missing_summary],
    ignore_index=True
)

missing_summary = missing_summary[["source", "field", "missing_rate"]]

missing_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/missing_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

missing_summary.to_csv(missing_summary_path, index=False)

display(missing_summary)

print("Saved missing summary to:", missing_summary_path)

,source,field,missing_rate
0,google_play,review_id,0.000000
1,google_play,user_name,0.000000
2,google_play,review_text,0.000000
3,google_play,rating,0.000000
4,google_play,thumbs_up,0.000000
5,google_play,review_created_version,0.182333
6,google_play,review_date,0.000000
7,google_play,developer_reply,0.670667
8,google_play,reply_date,0.670667
9,google_play,app_version,0.182333


Saved missing summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/missing_summary_run3_2026_06_23.csv


In [18]:
google_app_version_coverage = (
    google_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        total_reviews=("review_id", "count"),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum())
    )
    .reset_index()
)

ios_app_version_coverage = (
    ios_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        total_reviews=("review_id", "count"),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum())
    )
    .reset_index()
)

app_version_coverage = pd.concat(
    [google_app_version_coverage, ios_app_version_coverage],
    ignore_index=True
)

app_version_coverage["app_version_coverage_rate"] = (
    app_version_coverage["reviews_with_app_version"] / app_version_coverage["total_reviews"]
)

app_version_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/app_version_coverage_{RUN_LABEL}_{RUN_DATE}.csv"
)

app_version_coverage.to_csv(app_version_path, index=False)

display(app_version_coverage)

print("Saved app version coverage to:", app_version_path)

,source,app_name,total_reviews,reviews_with_app_version,app_version_coverage_rate
0,google_play,Spotify,2000,1656,0.8280
1,google_play,TikTok,2000,1293,0.6465
2,google_play,YouTube,2000,1957,0.9785
3,ios_app_store,Spotify,2400,2400,1.0000
4,ios_app_store,TikTok,1900,1900,1.0000
5,ios_app_store,YouTube,2500,2500,1.0000


Saved app version coverage to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/app_version_coverage_run3_2026_06_23.csv


In [19]:
def safe_detect_language(text):
    try:
        if pd.isna(text) or str(text).strip() == "":
            return "unknown"
        return detect(str(text))
    except:
        return "unknown"


google_reviews_checked["detected_language"] = google_reviews_checked["review_text"].apply(safe_detect_language)
ios_reviews_checked["detected_language"] = ios_reviews_checked["review_text"].apply(safe_detect_language)

google_language_summary = (
    google_reviews_checked
    .groupby(["source", "app_name", "detected_language"])
    .size()
    .reset_index(name="review_count")
)

ios_language_summary = (
    ios_reviews_checked
    .groupby(["source", "app_name", "detected_language"])
    .size()
    .reset_index(name="review_count")
)

language_summary = pd.concat(
    [google_language_summary, ios_language_summary],
    ignore_index=True
)

language_summary = language_summary.sort_values(
    ["source", "app_name", "review_count"],
    ascending=[True, True, False]
)

language_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/language_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

language_summary.to_csv(language_path, index=False)

display(language_summary)

print("Saved language summary to:", language_path)

,source,app_name,detected_language,review_count
7,google_play,Spotify,en,1429
29,google_play,Spotify,so,99
0,google_play,Spotify,af,57
26,google_play,Spotify,ro,34
16,google_play,Spotify,id,31
...,...,...,...,...
212,ios_app_store,YouTube,mk,1
213,ios_app_store,YouTube,ml,1
215,ios_app_store,YouTube,ne,1
228,ios_app_store,YouTube,ta,1


Saved language summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/language_summary_run3_2026_06_23.csv


In [20]:
google_checked_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/google_play_reviews_checked_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_checked_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/ios_reviews_checked_{RUN_LABEL}_{RUN_DATE}.csv"
)

google_reviews_checked.to_csv(google_checked_path, index=False)
ios_reviews_checked.to_csv(ios_checked_path, index=False)

print("Saved Google Play checked data to:", google_checked_path)
print("Saved iOS checked data to:", ios_checked_path)

Saved Google Play checked data to: /content/drive/MyDrive/app_review_source_validation/data/processed/google_play_reviews_checked_run3_2026_06_23.csv
Saved iOS checked data to: /content/drive/MyDrive/app_review_source_validation/data/processed/ios_reviews_checked_run3_2026_06_23.csv


## 8.1. Run 1 Findings

This clean Run 1 validation tested Google Play and iOS App Store public review sources for recurring review ingestion.

### Google Play Result

Google Play performed strongly in Run 1.

The collection returned 6,000 total reviews across three apps: YouTube, TikTok, and Spotify. Each app returned 2,000 newest US / English reviews. All three app requests completed successfully, with no failed app-level requests.

Google Play also provided stable review IDs. The duplicate check showed 6,000 total reviews, 6,000 unique review keys, and 0 duplicate rows. This is important because stable review IDs make future repeated collection and duplicate handling much easier.

The metadata coverage was also usable. Core fields such as review ID, user name, review text, rating, thumbs-up count, and review date were complete. App version fields were partially missing, and developer reply fields were missing for many reviews, but this is expected because not every review has an app version or developer response.

Based on Run 1, Google Play is the stronger primary source for repeated collection testing.

### iOS App Store Result

The iOS App Store public RSS feed was also technically accessible.

The Run 1 iOS test requested reviews for three apps across five countries and ten pages per app-country pair. The test collected 7,000 total reviews. Most app-country pairs returned usable results, but TikTok India returned 0 reviews with 10 empty pages.

This shows that iOS country and page rotation can increase review coverage, but the coverage is still less consistent than Google Play. Some app-country pairs may return empty results even when the request itself does not fail.

The iOS duplicate check showed 7,000 total reviews, 6,899 unique review keys, and 101 duplicate rows. The duplicate rate was about 1.44%. This is manageable, but it confirms that duplicate handling is necessary for iOS.

### Source Comparison

Google Play and iOS both passed the basic technical access test, but they have different strengths.

Google Play is cleaner for recurring ingestion because it returned stable review IDs, no duplicate rows in Run 1, strong volume, and consistent metadata.

iOS returned even more total reviews in this run, but it required country/page rotation and still had empty coverage for one app-country pair. iOS also had some duplicate rows, so it is better treated as a secondary source for now.

### Exploratory and Data Quality Findings

I also started exploratory and data quality analysis on the collected data.

The rating distribution shows that both sources contain a mix of positive and negative reviews, with many 1-star and 5-star reviews. This is useful for downstream sentiment or quality analysis.

The review length summary shows that iOS reviews are generally longer than Google Play reviews in this run. For example, TikTok iOS reviews had a higher average and median review length than TikTok Google Play reviews.

The missing field summary shows that Google Play core fields were complete, but app version and developer reply fields had missing values. iOS fields checked in this run were complete for the collected reviews.

The app version coverage check shows that iOS had full app version coverage in this run, while Google Play app version coverage varied by app.

The language detection result suggests that most reviews are English, but some reviews were detected as other languages. Since language detection can be noisy for short reviews, this should be treated as a rough data quality flag rather than a perfect language label.

### Current Conclusion

Run 1 confirms that Google Play is feasible as the primary source for a recurring app review ingestion pilot.

iOS App Store public RSS feed is also feasible as a secondary source, but it has more coverage and duplicate-handling issues.

The next step is to run the same notebook again as Run 2 and compare Run 2 against Run 1. The Run 2 comparison should focus on new review counts, overlap with Run 1, duplicate rates across runs, field consistency, request failures, and freshness patterns.

In [24]:
run1_report = f"""
# Run 1 Findings

## Purpose

This clean Run 1 validation tested Google Play and iOS App Store public review sources for recurring review ingestion.

The validation focused on:

- access method
- review volume
- available metadata fields
- pagination and batching
- duplicate handling
- freshness
- field consistency
- request failures
- exploratory analysis
- data quality issues

## Google Play Result

Google Play performed strongly in Run 1.

The collection returned 6,000 total reviews across three apps: YouTube, TikTok, and Spotify. Each app returned 2,000 newest US / English reviews. All three app requests completed successfully, with no failed app-level requests.

Google Play also provided stable review IDs. The duplicate check showed 6,000 total reviews, 6,000 unique review keys, and 0 duplicate rows. This is important because stable review IDs make future repeated collection and duplicate handling much easier.

The metadata coverage was also usable. Core fields such as review ID, user name, review text, rating, thumbs-up count, and review date were complete. App version fields were partially missing, and developer reply fields were missing for many reviews, but this is expected because not every review has an app version or developer response.

Based on Run 1, Google Play is the stronger primary source for repeated collection testing.

## iOS App Store Result

The iOS App Store public RSS feed was also technically accessible.

The Run 1 iOS test requested reviews for three apps across five countries and ten pages per app-country pair. The test collected 7,000 total reviews. Most app-country pairs returned usable results, but TikTok India returned 0 reviews with 10 empty pages.

This shows that iOS country and page rotation can increase review coverage, but the coverage is still less consistent than Google Play. Some app-country pairs may return empty results even when the request itself does not fail.

The iOS duplicate check showed 7,000 total reviews, 6,899 unique review keys, and 101 duplicate rows. The duplicate rate was about 1.44%. This is manageable, but it confirms that duplicate handling is necessary for iOS.

## Source Comparison

Google Play and iOS both passed the basic technical access test, but they have different strengths.

Google Play is cleaner for recurring ingestion because it returned stable review IDs, no duplicate rows in Run 1, strong volume, and consistent metadata.

iOS returned even more total reviews in this run, but it required country/page rotation and still had empty coverage for one app-country pair. iOS also had some duplicate rows, so it is better treated as a secondary source for now.

## Exploratory and Data Quality Findings

I also started exploratory and data quality analysis on the collected data.

The rating distribution shows that both sources contain a mix of positive and negative reviews, with many 1-star and 5-star reviews. This is useful for downstream sentiment or quality analysis.

The review length summary shows that iOS reviews are generally longer than Google Play reviews in this run.

The missing field summary shows that Google Play core fields were complete, but app version and developer reply fields had missing values. iOS fields checked in this run were complete for the collected reviews.

The app version coverage check shows that iOS had full app version coverage in this run, while Google Play app version coverage varied by app.

The language detection result suggests that most reviews are English, but some reviews were detected as other languages. Since language detection can be noisy for short reviews, this should be treated as a rough data quality flag rather than a perfect language label.

## Current Conclusion

Run 1 confirms that Google Play is feasible as the primary source for a recurring app review ingestion pilot.

iOS App Store public RSS feed is also feasible as a secondary source, but it has more coverage and duplicate-handling issues.

The next step is to run the same notebook again as Run 2 and compare Run 2 against Run 1. The Run 2 comparison should focus on new review counts, overlap with Run 1, duplicate rates across runs, field consistency, request failures, and freshness patterns.
"""

run1_report_path = os.path.join(
    PROJECT_DIR,
    f"reports/run1_findings_{RUN_DATE}.md"
)

with open(run1_report_path, "w") as f:
    f.write(run1_report)

print("Saved Run 1 report to:", run1_report_path)

Saved Run 1 report to: /content/drive/MyDrive/app_review_source_validation/reports/run1_findings_2026_06_23.md


## 8.2. Run 2 Findings

This notebook records Run 2 of the public app review source validation.

### Google Play Result

Google Play remained stable in Run 2.

The collection again returned 6,000 total reviews across YouTube, TikTok, and Spotify, with 2,000 newest US / English reviews per app. There were no failed app-level requests. The duplicate check again showed 6,000 total reviews, 6,000 unique review keys, and 0 duplicate rows.

This confirms that Google Play collection is stable across repeated runs.

### iOS App Store Result

The iOS App Store public RSS feed was also stable in Run 2.

The Run 2 iOS test again collected 7,000 total reviews across three apps and five countries. Similar to Run 1, TikTok India returned 0 reviews, while the other app-country pairs returned usable results.

The iOS duplicate check showed 7,000 total reviews, 6,891 unique review keys, and 109 duplicate rows. This confirms that iOS duplicate handling is necessary.

### Run 1 vs Run 2 Comparison

The repeated-run comparison provides useful evidence for recurring ingestion feasibility.

For Google Play, Run 2 captured both overlapping reviews and new reviews compared with Run 1. This is expected for newest-review collection and is a positive sign for recurring ingestion.

In the Google Play comparison:

- YouTube had 1,238 new reviews in Run 2 and 762 overlapping reviews with Run 1.
- TikTok had 612 new reviews in Run 2 and 1,388 overlapping reviews with Run 1.
- Spotify had 695 new reviews in Run 2 and 1,305 overlapping reviews with Run 1.

All three Google Play app comparisons showed the same schema across runs.

For iOS, Run 2 also captured both overlapping and new reviews at the app-country level. Most app-country pairs had the same schema across runs. Some combinations had high overlap, while several US pairs had more new reviews, such as YouTube US, TikTok US, and Spotify US.

This suggests that iOS can also support repeated collection, but it has more duplicate and coverage issues than Google Play. TikTok India remained unavailable in this run, which confirms that some iOS app-country pairs can return empty results even when the request itself does not fail.

### Current Conclusion

After two runs, Google Play remains the strongest candidate source for a recurring app review ingestion pilot.

It provides:

- stable repeated collection
- strong review volume
- stable review IDs
- clean duplicate handling
- cross-run schema consistency
- evidence of new review capture across runs

iOS App Store public RSS remains technically usable, but it is still better treated as a secondary source because of duplicate rows, country/page dependence, and less consistent coverage.

The next step is to upload Run 2 outputs to GitHub and share the repeated-run findings with John.

In [22]:
run2_report = """
# Run 2 Findings

## Purpose

This notebook records Run 2 of the public app review source validation.

## Google Play Result

Google Play remained stable in Run 2.

The collection again returned 6,000 total reviews across YouTube, TikTok, and Spotify, with 2,000 newest US / English reviews per app. There were no failed app-level requests. The duplicate check again showed 6,000 total reviews, 6,000 unique review keys, and 0 duplicate rows.

This confirms that Google Play collection is stable across repeated runs.

## iOS App Store Result

The iOS App Store public RSS feed was also stable in Run 2.

The Run 2 iOS test again collected 7,000 total reviews across three apps and five countries. Similar to Run 1, TikTok India returned 0 reviews, while the other app-country pairs returned usable results.

The iOS duplicate check showed 7,000 total reviews, 6,891 unique review keys, and 109 duplicate rows. This confirms that iOS duplicate handling is necessary.

## Run 1 vs Run 2 Comparison

The repeated-run comparison provides useful evidence for recurring ingestion feasibility.

For Google Play:

- YouTube had 1,238 new reviews in Run 2 and 762 overlapping reviews with Run 1.
- TikTok had 612 new reviews in Run 2 and 1,388 overlapping reviews with Run 1.
- Spotify had 695 new reviews in Run 2 and 1,305 overlapping reviews with Run 1.

All three Google Play app comparisons showed the same schema across runs.

For iOS, Run 2 also captured both overlapping and new reviews at the app-country level. Most app-country pairs had the same schema across runs. Some combinations had high overlap, while several US pairs had more new reviews, such as YouTube US, TikTok US, and Spotify US.

TikTok India remained unavailable in this run, which confirms that some iOS app-country pairs can return empty results even when the request itself does not fail.

## Current Conclusion

After two runs, Google Play remains the strongest candidate source for a recurring app review ingestion pilot.

It provides:

- stable repeated collection
- strong review volume
- stable review IDs
- clean duplicate handling
- cross-run schema consistency
- evidence of new review capture across runs

iOS App Store public RSS remains technically usable, but it is still better treated as a secondary source because of duplicate rows, country/page dependence, and less consistent coverage.

The next step is to upload Run 2 outputs to GitHub and share the repeated-run findings with John.
"""

run2_report_path = os.path.join(
    PROJECT_DIR,
    f"reports/run2_findings_{RUN_DATE}.md"
)

with open(run2_report_path, "w") as f:
    f.write(run2_report)

print("Saved Run 2 report to:", run2_report_path)

Saved Run 2 report to: /content/drive/MyDrive/app_review_source_validation/reports/run2_findings_2026_06_23.md


## 8.3. Run 3 Findings

This notebook records Run 3 of the public app review source validation.

### Google Play Result

Google Play remained stable in Run 3.

The collection again returned 6,000 total reviews across YouTube, TikTok, and Spotify, with 2,000 newest US / English reviews per app. There were no failed app-level requests.

The duplicate check again showed stable review keys and no duplicate rows within the Google Play Run 3 collection.

This confirms that Google Play collection continues to be stable across repeated runs.

### iOS App Store Result

The iOS App Store public RSS feed was also technically accessible in Run 3.

The Run 3 iOS test repeated the same structure as the previous runs: three apps, five countries, and ten pages per app-country pair.

Most app-country pairs continued to return usable reviews. Similar to earlier runs, some iOS app-country combinations were less reliable, which confirms that iOS coverage depends more on country and page rotation.

The iOS results also continued to show more duplicate handling needs than Google Play.

### Exploratory and Data Quality Checks

Run 3 repeated the same exploratory and data quality checks as Run 1 and Run 2, including:

* rating distribution
* review length
* missing fields
* app version coverage
* language detection
* duplicate summary
* source-level comparison

The results stayed generally consistent with the earlier runs. Google Play had stronger within-run uniqueness and cleaner repeated collection behavior. iOS remained technically usable, but still had more coverage and duplicate-handling issues.

### Current Run 3 Conclusion

Run 3 further supports the same conclusion from Run 1 and Run 2.

Google Play remains the stronger primary source for a recurring app review ingestion pilot because it provides stable volume, stable review IDs, clean duplicate handling, and consistent metadata structure.

iOS App Store public RSS remains useful as a secondary source, but it should not be treated as the primary source because of country/page dependence, empty-page risk, and more duplicate rows.

The next step is to compare Run 3 with Run 2 to confirm repeated-run stability, new review capture, schema consistency, and freshness patterns.


In [26]:
run3_report = """
# Run 3 Findings

## Purpose

This notebook records Run 3 of the public app review source validation.

## Google Play Result

Google Play remained stable in Run 3.

The collection again returned 6,000 total reviews across YouTube, TikTok, and Spotify, with 2,000 newest US / English reviews per app. There were no failed app-level requests.

The duplicate check again showed stable review keys and no duplicate rows within the Google Play Run 3 collection.

This confirms that Google Play collection continues to be stable across repeated runs.

## iOS App Store Result

The iOS App Store public RSS feed was also technically accessible in Run 3.

The Run 3 iOS test repeated the same structure as the previous runs: three apps, five countries, and ten pages per app-country pair.

Most app-country pairs continued to return usable reviews. Similar to earlier runs, some iOS app-country combinations were less reliable, which confirms that iOS coverage depends more on country and page rotation.

The iOS results also continued to show more duplicate handling needs than Google Play.

## Exploratory and Data Quality Checks

Run 3 repeated the same exploratory and data quality checks as Run 1 and Run 2, including:

- rating distribution
- review length
- missing fields
- app version coverage
- language detection
- duplicate summary
- source-level comparison

The results stayed generally consistent with the earlier runs. Google Play had stronger within-run uniqueness and cleaner repeated collection behavior. iOS remained technically usable, but still had more coverage and duplicate-handling issues.

## Current Run 3 Conclusion

Run 3 further supports the same conclusion from Run 1 and Run 2.

Google Play remains the stronger primary source for a recurring app review ingestion pilot because it provides stable volume, stable review IDs, clean duplicate handling, and consistent metadata structure.

iOS App Store public RSS remains useful as a secondary source, but it should not be treated as the primary source because of country/page dependence, empty-page risk, and more duplicate rows.

The next step is to compare Run 3 with Run 2 to confirm repeated-run stability, new review capture, schema consistency, and freshness patterns.
"""

run3_report_path = os.path.join(
    PROJECT_DIR,
    f"reports/run3_findings_{RUN_DATE}.md"
)

with open(run3_report_path, "w") as f:
    f.write(run3_report)

print("Saved Run 3 report to:", run3_report_path)

Saved Run 3 report to: /content/drive/MyDrive/app_review_source_validation/reports/run3_findings_2026_06_23.md


## 9. Run 2 vs Run 3 Comparison

This section compares Run 3 with Run 2 to evaluate whether repeated collection remains stable across another run.

The comparison checks:

* new review counts
* overlap with the previous run
* duplicate rate across runs
* field consistency
* newest review timestamp
* request stability

Google Play is the main focus because it is the primary candidate source. iOS is also compared at the app-country level because its coverage depends more on country and page combinations.


In [27]:
RUN2_DATE = "2026_06_22"
RUN3_DATE = "2026_06_23"

run2_google_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/google_play_reviews_checked_run2_{RUN2_DATE}.csv"
)

run3_google_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/google_play_reviews_checked_run3_{RUN3_DATE}.csv"
)

run2_ios_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/ios_reviews_checked_run2_{RUN2_DATE}.csv"
)

run3_ios_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/ios_reviews_checked_run3_{RUN3_DATE}.csv"
)

print("Run 2 Google path exists:", os.path.exists(run2_google_path))
print("Run 3 Google path exists:", os.path.exists(run3_google_path))
print("Run 2 iOS path exists:", os.path.exists(run2_ios_path))
print("Run 3 iOS path exists:", os.path.exists(run3_ios_path))

print(run2_google_path)
print(run3_google_path)
print(run2_ios_path)
print(run3_ios_path)

Run 2 Google path exists: True
Run 3 Google path exists: True
Run 2 iOS path exists: True
Run 3 iOS path exists: True
/content/drive/MyDrive/app_review_source_validation/data/processed/google_play_reviews_checked_run2_2026_06_22.csv
/content/drive/MyDrive/app_review_source_validation/data/processed/google_play_reviews_checked_run3_2026_06_23.csv
/content/drive/MyDrive/app_review_source_validation/data/processed/ios_reviews_checked_run2_2026_06_22.csv
/content/drive/MyDrive/app_review_source_validation/data/processed/ios_reviews_checked_run3_2026_06_23.csv


In [28]:
run2_google = pd.read_csv(run2_google_path)
run3_google = pd.read_csv(run3_google_path)

google_comparison_rows = []

for app in run3_google["app_name"].unique():
    r2 = run2_google[run2_google["app_name"] == app]
    r3 = run3_google[run3_google["app_name"] == app]

    r2_keys = set(r2["review_key"])
    r3_keys = set(r3["review_key"])

    new_reviews = r3_keys - r2_keys
    overlapping_reviews = r3_keys & r2_keys

    google_comparison_rows.append({
        "source": "google_play",
        "app_name": app,
        "run2_count": len(r2),
        "run3_count": len(r3),
        "new_reviews_in_run3": len(new_reviews),
        "overlap_with_run2": len(overlapping_reviews),
        "run3_overlap_rate": len(overlapping_reviews) / len(r3) if len(r3) > 0 else None,
        "run2_newest_review": r2["review_date"].max(),
        "run3_newest_review": r3["review_date"].max(),
        "schema_same": set(run2_google.columns) == set(run3_google.columns)
    })

google_run2_vs_run3_comparison = pd.DataFrame(google_comparison_rows)

google_run2_vs_run3_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_run2_vs_run3_comparison_{RUN3_DATE}.csv"
)

google_run2_vs_run3_comparison.to_csv(google_run2_vs_run3_path, index=False)

display(google_run2_vs_run3_comparison)

print("Saved Google Play Run 2 vs Run 3 comparison to:", google_run2_vs_run3_path)

,source,app_name,run2_count,run3_count,new_reviews_in_run3,overlap_with_run2,run3_overlap_rate,run2_newest_review,run3_newest_review,schema_same
0,google_play,YouTube,2000,2000,1402,598,0.299,2026-06-21 21:07:36,2026-06-22 18:36:47,True
1,google_play,TikTok,2000,2000,596,1404,0.702,2026-06-21 21:16:36,2026-06-22 18:37:36,True
2,google_play,Spotify,2000,2000,694,1306,0.653,2026-06-21 21:00:03,2026-06-22 18:36:20,True


Saved Google Play Run 2 vs Run 3 comparison to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_run2_vs_run3_comparison_2026_06_23.csv


In [29]:
run2_ios = pd.read_csv(run2_ios_path)
run3_ios = pd.read_csv(run3_ios_path)

ios_comparison_rows = []

for app in run3_ios["app_name"].unique():
    countries_for_app = run3_ios[run3_ios["app_name"] == app]["country"].unique()

    for country in countries_for_app:
        r2 = run2_ios[
            (run2_ios["app_name"] == app) &
            (run2_ios["country"] == country)
        ]

        r3 = run3_ios[
            (run3_ios["app_name"] == app) &
            (run3_ios["country"] == country)
        ]

        r2_keys = set(r2["review_key"])
        r3_keys = set(r3["review_key"])

        new_reviews = r3_keys - r2_keys
        overlapping_reviews = r3_keys & r2_keys

        ios_comparison_rows.append({
            "source": "ios_app_store",
            "app_name": app,
            "country": country,
            "run2_count": len(r2),
            "run3_count": len(r3),
            "new_reviews_in_run3": len(new_reviews),
            "overlap_with_run2": len(overlapping_reviews),
            "run3_overlap_rate": len(overlapping_reviews) / len(r3) if len(r3) > 0 else None,
            "run2_newest_review": r2["review_date"].max() if len(r2) > 0 else None,
            "run3_newest_review": r3["review_date"].max() if len(r3) > 0 else None,
            "schema_same": set(run2_ios.columns) == set(run3_ios.columns)
        })

ios_run2_vs_run3_comparison = pd.DataFrame(ios_comparison_rows)

ios_run2_vs_run3_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_run2_vs_run3_comparison_{RUN3_DATE}.csv"
)

ios_run2_vs_run3_comparison.to_csv(ios_run2_vs_run3_path, index=False)

display(ios_run2_vs_run3_comparison)

print("Saved iOS Run 2 vs Run 3 comparison to:", ios_run2_vs_run3_path)

,source,app_name,country,run2_count,run3_count,new_reviews_in_run3,overlap_with_run2,run3_overlap_rate,run2_newest_review,run3_newest_review,schema_same
0,ios_app_store,YouTube,us,500,500,174,326,0.652000,2026-06-21T08:44:04-07:00,2026-06-22T04:26:11-07:00,True
1,ios_app_store,YouTube,gb,500,500,11,489,0.978000,2026-06-21T08:42:29-07:00,2026-06-22T04:00:19-07:00,True
2,ios_app_store,YouTube,ca,500,500,21,479,0.958000,2026-06-21T08:25:54-07:00,2026-06-22T02:43:35-07:00,True
3,ios_app_store,YouTube,au,500,500,5,495,0.990000,2026-06-21T07:53:13-07:00,2026-06-22T04:11:01-07:00,True
4,ios_app_store,YouTube,in,500,500,13,487,0.974000,2026-06-21T07:41:18-07:00,2026-06-22T01:42:36-07:00,True
5,ios_app_store,TikTok,us,500,450,113,337,0.748889,2026-06-21T08:42:55-07:00,2026-06-22T04:18:31-07:00,True
6,ios_app_store,TikTok,gb,500,450,100,350,0.777778,2026-06-21T08:19:37-07:00,2026-06-22T03:42:50-07:00,True
7,ios_app_store,TikTok,ca,500,500,11,439,0.878000,2026-06-21T05:29:05-07:00,2026-06-21T22:00:04-07:00,True
8,ios_app_store,TikTok,au,500,500,168,332,0.664000,2026-06-12T18:23:20-07:00,2026-06-22T02:08:41-07:00,True
9,ios_app_store,Spotify,us,500,500,310,190,0.380000,2026-06-21T08:41:06-07:00,2026-06-22T04:39:51-07:00,True


Saved iOS Run 2 vs Run 3 comparison to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_run2_vs_run3_comparison_2026_06_23.csv


## Run 2 vs Run 3 Comparison Findings

The Run 2 vs Run 3 comparison gives another repeated-run check for both Google Play and iOS App Store public review sources.

### Google Play Findings

Google Play continued to perform strongly in the Run 2 vs Run 3 comparison.

Each app again returned 2,000 reviews in Run 3, and the schema stayed the same across runs for all three apps.

Compared with Run 2:

- YouTube captured 1,402 new reviews in Run 3, with 598 overlapping reviews.
- TikTok captured 596 new reviews in Run 3, with 1,404 overlapping reviews.
- Spotify captured 694 new reviews in Run 3, with 1,306 overlapping reviews.

This is a good result for repeated ingestion. It shows that Google Play can keep returning a stable batch size while also capturing new reviews across runs. The overlap is expected because the collection pulls the newest reviews, so some recent reviews from the previous run will still appear in the next run.

The schema consistency also supports using Google Play as the primary source for a recurring ingestion pipeline.

### iOS App Store Findings

The iOS App Store public RSS feed also remained technically usable in Run 3.

Most app-country pairs returned reviews again, and the schema stayed consistent across the repeated comparison. Some app-country pairs had high overlap with Run 2, while others captured more new reviews.

For example:

- YouTube US captured 174 new reviews in Run 3.
- TikTok US captured 113 new reviews in Run 3.
- TikTok Australia captured 168 new reviews in Run 3.
- Spotify US captured 310 new reviews in Run 3.

However, iOS still showed more instability than Google Play. Some app-country pairs returned fewer reviews in Run 3, such as TikTok US and TikTok UK returning 450 reviews instead of 500. This confirms that iOS coverage depends more on country/page combinations and is less consistent than Google Play.

### Run 2 vs Run 3 Conclusion

The Run 2 vs Run 3 comparison supports the same conclusion as the earlier runs.

Google Play is stable, repeatable, and better suited as the primary source. It consistently returns strong review volume, stable schema, no within-run duplicate issue, and clear new review capture across runs.

iOS is also technically repeatable, but it has more coverage variation and duplicate-handling needs. It can remain a useful secondary source, especially for additional country-level coverage, but it should not be the main source for the recurring ingestion pilot.

Overall, Run 3 strengthens the conclusion that Google Play is the best primary path for this module, while iOS should stay as a secondary validation path.

## 10. Three-Run Module Summary

This section summarizes the completed Run 1, Run 2, and Run 3 results to wrap up the current source validation module.

The goal is not to compare every review across all three runs directly. Instead, this section synthesizes the repeated-run evidence from the module:

- source stability across runs
- review volume consistency
- duplicate handling
- new review capture
- schema consistency
- iOS country/page coverage limitations
- final source recommendation

Run 1 vs Run 2 and Run 2 vs Run 3 comparisons are used as the repeated-run evidence.

In [30]:
RUN1_DATE = "2026_06_22"
RUN2_DATE = "2026_06_22"
RUN3_DATE = "2026_06_23"

run_dates = {
    "run1": RUN1_DATE,
    "run2": RUN2_DATE,
    "run3": RUN3_DATE
}

source_summary_frames = []

for run_label, run_date in run_dates.items():
    path = os.path.join(
        PROJECT_DIR,
        f"outputs/summaries/source_comparison_{run_label}_{run_date}.csv"
    )

    temp = pd.read_csv(path)
    temp["run"] = run_label
    source_summary_frames.append(temp)

three_run_source_summary = pd.concat(source_summary_frames, ignore_index=True)

three_run_source_summary = three_run_source_summary[
    [
        "run",
        "source",
        "total_reviews",
        "unique_review_keys",
        "duplicate_rows",
        "duplicate_rate",
        "failed_requests_or_apps",
        "main_limitation"
    ]
]

three_run_source_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/three_run_source_summary_{RUN3_DATE}.csv"
)

three_run_source_summary.to_csv(three_run_source_summary_path, index=False)

display(three_run_source_summary)

print("Saved three-run source summary to:", three_run_source_summary_path)

,run,source,total_reviews,unique_review_keys,duplicate_rows,duplicate_rate,failed_requests_or_apps,main_limitation
0,run1,google_play,6000,6000,0,0.000000,0,Needs repeated runs to confirm freshness and n...
1,run1,ios_app_store,7000,6899,101,0.014429,0,Country/page coverage can be inconsistent; som...
2,run2,google_play,6000,6000,0,0.000000,0,Needs repeated runs to confirm freshness and n...
3,run2,ios_app_store,7000,6795,205,0.029286,0,Country/page coverage can be inconsistent; som...
4,run3,google_play,6000,6000,0,0.000000,0,Needs repeated runs to confirm freshness and n...
5,run3,ios_app_store,6800,6750,50,0.007353,0,Country/page coverage can be inconsistent; som...


Saved three-run source summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/three_run_source_summary_2026_06_23.csv


In [32]:
google_run1_vs_run2_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_run1_vs_run2_comparison_{RUN2_DATE}.csv"
)

google_run2_vs_run3_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_run2_vs_run3_comparison_{RUN3_DATE}.csv"
)

google_12 = pd.read_csv(google_run1_vs_run2_path)
google_23 = pd.read_csv(google_run2_vs_run3_path)

google_12_clean = google_12.rename(columns={
    "run1_count": "previous_run_count",
    "run2_count": "current_run_count",
    "new_reviews_in_run2": "new_reviews_in_current_run",
    "overlap_with_run1": "overlap_with_previous_run",
    "run2_overlap_rate": "current_run_overlap_rate",
    "run1_newest_review": "previous_run_newest_review",
    "run2_newest_review": "current_run_newest_review"
})

google_12_clean["comparison_pair"] = "run1_vs_run2"

google_23_clean = google_23.rename(columns={
    "run2_count": "previous_run_count",
    "run3_count": "current_run_count",
    "new_reviews_in_run3": "new_reviews_in_current_run",
    "overlap_with_run2": "overlap_with_previous_run",
    "run3_overlap_rate": "current_run_overlap_rate",
    "run2_newest_review": "previous_run_newest_review",
    "run3_newest_review": "current_run_newest_review"
})

google_23_clean["comparison_pair"] = "run2_vs_run3"

google_repeated_run_summary = pd.concat(
    [google_12_clean, google_23_clean],
    ignore_index=True
)

google_repeated_run_summary = google_repeated_run_summary[
    [
        "comparison_pair",
        "source",
        "app_name",
        "previous_run_count",
        "current_run_count",
        "new_reviews_in_current_run",
        "overlap_with_previous_run",
        "current_run_overlap_rate",
        "previous_run_newest_review",
        "current_run_newest_review",
        "schema_same"
    ]
]

google_repeated_run_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_repeated_run_summary_{RUN3_DATE}.csv"
)

google_repeated_run_summary.to_csv(google_repeated_run_summary_path, index=False)

display(google_repeated_run_summary)

print("Saved Google Play repeated-run summary to:", google_repeated_run_summary_path)

,comparison_pair,source,app_name,previous_run_count,current_run_count,new_reviews_in_current_run,overlap_with_previous_run,current_run_overlap_rate,previous_run_newest_review,current_run_newest_review,schema_same
0,run1_vs_run2,google_play,YouTube,2000,2000,1238,762,0.3810,2026-06-21 02:48:36,2026-06-21 21:07:36,True
1,run1_vs_run2,google_play,TikTok,2000,2000,620,1380,0.6900,2026-06-21 02:49:34,2026-06-21 21:16:36,True
2,run1_vs_run2,google_play,Spotify,2000,2000,695,1305,0.6525,2026-06-21 02:49:56,2026-06-21 21:00:03,True
3,run2_vs_run3,google_play,YouTube,2000,2000,1402,598,0.2990,2026-06-21 21:07:36,2026-06-22 18:36:47,True
4,run2_vs_run3,google_play,TikTok,2000,2000,596,1404,0.7020,2026-06-21 21:16:36,2026-06-22 18:37:36,True
5,run2_vs_run3,google_play,Spotify,2000,2000,694,1306,0.6530,2026-06-21 21:00:03,2026-06-22 18:36:20,True


Saved Google Play repeated-run summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_repeated_run_summary_2026_06_23.csv


In [33]:
ios_run1_vs_run2_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_run1_vs_run2_comparison_{RUN2_DATE}.csv"
)

ios_run2_vs_run3_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_run2_vs_run3_comparison_{RUN3_DATE}.csv"
)

ios_12 = pd.read_csv(ios_run1_vs_run2_path)
ios_23 = pd.read_csv(ios_run2_vs_run3_path)

ios_12_clean = ios_12.rename(columns={
    "run1_count": "previous_run_count",
    "run2_count": "current_run_count",
    "new_reviews_in_run2": "new_reviews_in_current_run",
    "overlap_with_run1": "overlap_with_previous_run",
    "run2_overlap_rate": "current_run_overlap_rate",
    "run1_newest_review": "previous_run_newest_review",
    "run2_newest_review": "current_run_newest_review"
})

ios_12_clean["comparison_pair"] = "run1_vs_run2"

ios_23_clean = ios_23.rename(columns={
    "run2_count": "previous_run_count",
    "run3_count": "current_run_count",
    "new_reviews_in_run3": "new_reviews_in_current_run",
    "overlap_with_run2": "overlap_with_previous_run",
    "run3_overlap_rate": "current_run_overlap_rate",
    "run2_newest_review": "previous_run_newest_review",
    "run3_newest_review": "current_run_newest_review"
})

ios_23_clean["comparison_pair"] = "run2_vs_run3"

ios_repeated_run_summary = pd.concat(
    [ios_12_clean, ios_23_clean],
    ignore_index=True
)

ios_repeated_run_summary = ios_repeated_run_summary[
    [
        "comparison_pair",
        "source",
        "app_name",
        "country",
        "previous_run_count",
        "current_run_count",
        "new_reviews_in_current_run",
        "overlap_with_previous_run",
        "current_run_overlap_rate",
        "previous_run_newest_review",
        "current_run_newest_review",
        "schema_same"
    ]
]

ios_repeated_run_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_repeated_run_summary_{RUN3_DATE}.csv"
)

ios_repeated_run_summary.to_csv(ios_repeated_run_summary_path, index=False)

display(ios_repeated_run_summary)

print("Saved iOS repeated-run summary to:", ios_repeated_run_summary_path)

,comparison_pair,source,app_name,country,previous_run_count,current_run_count,new_reviews_in_current_run,overlap_with_previous_run,current_run_overlap_rate,previous_run_newest_review,current_run_newest_review,schema_same
0,run1_vs_run2,ios_app_store,YouTube,us,500,500,117,382,0.764000,2026-06-20T15:58:19-07:00,2026-06-21T08:44:04-07:00,True
1,run1_vs_run2,ios_app_store,YouTube,gb,500,500,17,483,0.966000,2026-06-20T14:23:30-07:00,2026-06-21T08:42:29-07:00,True
2,run1_vs_run2,ios_app_store,YouTube,ca,500,500,17,483,0.966000,2026-06-20T15:38:26-07:00,2026-06-21T08:25:54-07:00,True
3,run1_vs_run2,ios_app_store,YouTube,au,500,500,11,489,0.978000,2026-06-20T14:24:32-07:00,2026-06-21T07:53:13-07:00,True
4,run1_vs_run2,ios_app_store,YouTube,in,500,500,15,485,0.970000,2026-06-20T15:15:21-07:00,2026-06-21T07:41:18-07:00,True
5,run1_vs_run2,ios_app_store,TikTok,us,500,500,205,288,0.576000,2026-06-20T15:57:12-07:00,2026-06-21T08:42:55-07:00,True
6,run1_vs_run2,ios_app_store,TikTok,gb,500,500,16,386,0.772000,2026-06-20T14:41:14-07:00,2026-06-21T08:19:37-07:00,True
7,run1_vs_run2,ios_app_store,TikTok,ca,500,500,54,446,0.892000,2026-06-20T12:22:31-07:00,2026-06-21T05:29:05-07:00,True
8,run1_vs_run2,ios_app_store,TikTok,au,500,500,63,338,0.676000,2026-06-12T03:46:26-07:00,2026-06-12T18:23:20-07:00,True
9,run1_vs_run2,ios_app_store,Spotify,us,500,500,197,303,0.606000,2026-06-20T15:55:38-07:00,2026-06-21T08:41:06-07:00,True


Saved iOS repeated-run summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_repeated_run_summary_2026_06_23.csv


In [34]:
module_conclusion_summary = pd.DataFrame([
    {
        "source": "google_play",
        "access_result": "passed",
        "repeated_collection_result": "stable across three runs",
        "volume_result": "6,000 reviews collected in each run; 2,000 reviews per app",
        "duplicate_result": "0 duplicate rows within each run",
        "schema_result": "schema stayed consistent across repeated comparisons",
        "freshness_result": "new reviews captured in both Run 2 and Run 3 comparisons",
        "main_limitation": "needs continued scheduled runs if production-level freshness monitoring is required",
        "recommendation": "primary source"
    },
    {
        "source": "ios_app_store",
        "access_result": "passed",
        "repeated_collection_result": "technically repeatable but less consistent than Google Play",
        "volume_result": "useful volume, but dependent on app-country-page combinations",
        "duplicate_result": "duplicate rows appeared in each run and require handling",
        "schema_result": "schema stayed mostly consistent across repeated comparisons",
        "freshness_result": "new reviews captured in multiple app-country pairs",
        "main_limitation": "country/page coverage inconsistency, empty app-country pairs, and higher duplicate handling needs",
        "recommendation": "secondary source"
    }
])

module_conclusion_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/module_conclusion_summary_{RUN3_DATE}.csv"
)

module_conclusion_summary.to_csv(module_conclusion_summary_path, index=False)

display(module_conclusion_summary)

print("Saved module conclusion summary to:", module_conclusion_summary_path)

,source,access_result,repeated_collection_result,volume_result,duplicate_result,schema_result,freshness_result,main_limitation,recommendation
0,google_play,passed,stable across three runs,"6,000 reviews collected in each run; 2,000 rev...",0 duplicate rows within each run,schema stayed consistent across repeated compa...,new reviews captured in both Run 2 and Run 3 c...,needs continued scheduled runs if production-l...,primary source
1,ios_app_store,passed,technically repeatable but less consistent tha...,"useful volume, but dependent on app-country-pa...",duplicate rows appeared in each run and requir...,schema stayed mostly consistent across repeate...,new reviews captured in multiple app-country p...,"country/page coverage inconsistency, empty app...",secondary source


Saved module conclusion summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/module_conclusion_summary_2026_06_23.csv


## 11. Synthesized Summary and Final Module Conclusion

This section wraps up the current source validation module after three completed runs.

The goal of this module was to validate whether public app review sources can support a recurring review ingestion workflow. The validation focused on access method, review volume, metadata fields, pagination and batching, duplicate handling, freshness, schema consistency, request stability, and basic exploratory/data quality checks.

### Overall Google Play Result

Google Play was the strongest and most stable source across all three runs.

Across Run 1, Run 2, and Run 3, Google Play consistently returned:

* 6,000 reviews per run
* 2,000 reviews per app
* 3 tested apps: YouTube, TikTok, and Spotify
* 0 duplicate rows within each run
* stable review IDs
* consistent schema across repeated comparisons
* fresh reviews captured in repeated runs

The repeated-run comparisons showed that Google Play can capture new reviews while still keeping enough overlap with the previous run for duplicate handling.

In the Run 1 vs Run 2 comparison:

* YouTube captured 1,238 new reviews in Run 2, with 762 overlapping reviews.
* TikTok captured 620 new reviews in Run 2, with 1,380 overlapping reviews.
* Spotify captured 695 new reviews in Run 2, with 1,305 overlapping reviews.

In the Run 2 vs Run 3 comparison:

* YouTube captured 1,402 new reviews in Run 3, with 598 overlapping reviews.
* TikTok captured 596 new reviews in Run 3, with 1,404 overlapping reviews.
* Spotify captured 694 new reviews in Run 3, with 1,306 overlapping reviews.

All Google Play repeated-run comparisons showed the same schema across runs.

This confirms that Google Play is technically feasible for a small recurring review ingestion pilot. It can repeatedly return a stable review volume, preserve stable review IDs, support duplicate handling, and capture new reviews over time.

### Overall iOS App Store Result

The iOS App Store public RSS feed was also technically accessible and repeatable, but it was less consistent than Google Play.

Across the three runs, iOS returned useful review volume, but the exact results varied more:

* Run 1 collected 7,000 iOS reviews.
* Run 2 collected 7,000 iOS reviews.
* Run 3 collected 6,800 iOS reviews.

The duplicate counts also varied across runs. This confirms that duplicate handling is necessary for iOS.

The iOS repeated-run comparisons showed that iOS can capture new reviews across app-country pairs. However, iOS depends more heavily on country and page combinations. Some app-country pairs had high overlap across runs, while others captured more new reviews. Some combinations also returned fewer reviews in Run 3, which shows that iOS coverage is less consistent than Google Play.

Compared with Google Play, iOS has stronger limitations around:

* country/page coverage
* empty or incomplete app-country pairs
* duplicate rows
* less consistent collected volume

Based on these results, iOS should remain a secondary source rather than the primary ingestion path.

### Exploratory and Data Quality Findings

The exploratory and data quality checks were consistent with the source-level findings.

The analysis included:

* rating distribution
* review length
* missing fields
* app version coverage
* language detection
* duplicate summaries
* source-level comparison

Main findings:

* Both sources contain a mix of positive and negative reviews.
* Many reviews are either 1-star or 5-star.
* iOS reviews were generally longer than Google Play reviews.
* Google Play core fields were complete.
* Google Play app version and developer reply fields had missing values, which is expected.
* iOS checked fields were mostly complete for collected reviews.
* Language detection showed mostly English reviews, but some reviews were detected as other languages.
* Language detection should be treated as a rough data quality flag because short reviews can be misclassified.

### Final Recommendation

After three runs, Google Play should be treated as the primary source for the recurring app review ingestion pilot.

Google Play is recommended as the primary path because it provides:

* stable repeated collection
* strong and consistent review volume
* stable review IDs
* clean within-run duplicate handling
* schema consistency across runs
* clear new review capture across repeated runs
* usable metadata for downstream analysis

iOS App Store public RSS should remain a secondary source.

iOS is useful for additional coverage, especially across countries, but it should not be the main source because it has more duplicate rows, stronger country/page dependence, and less consistent collection volume.

### Final Module Conclusion

The current module validates that public app review collection is feasible.

The strongest path is to use Google Play as the main recurring ingestion source and keep iOS as a supplementary source. If this work moves into the next stage, the recommended next step is to convert the notebook workflow into reusable scripts and schedule additional recurring runs to continue monitoring freshness, failures, schema stability, and duplicate behavior over time.


In [35]:
synthesized_report = """
# Synthesized Summary and Final Module Conclusion

## Purpose

This report wraps up the current source validation module after three completed runs.

The goal of this module was to validate whether public app review sources can support a recurring review ingestion workflow. The validation focused on access method, review volume, metadata fields, pagination and batching, duplicate handling, freshness, schema consistency, request stability, and basic exploratory/data quality checks.

## Overall Google Play Result

Google Play was the strongest and most stable source across all three runs.

Across Run 1, Run 2, and Run 3, Google Play consistently returned:

- 6,000 reviews per run
- 2,000 reviews per app
- 3 tested apps: YouTube, TikTok, and Spotify
- 0 duplicate rows within each run
- stable review IDs
- consistent schema across repeated comparisons
- fresh reviews captured in repeated runs

The repeated-run comparisons showed that Google Play can capture new reviews while still keeping enough overlap with the previous run for duplicate handling.

In the Run 1 vs Run 2 comparison:

- YouTube captured 1,238 new reviews in Run 2, with 762 overlapping reviews.
- TikTok captured 620 new reviews in Run 2, with 1,380 overlapping reviews.
- Spotify captured 695 new reviews in Run 2, with 1,305 overlapping reviews.

In the Run 2 vs Run 3 comparison:

- YouTube captured 1,402 new reviews in Run 3, with 598 overlapping reviews.
- TikTok captured 596 new reviews in Run 3, with 1,404 overlapping reviews.
- Spotify captured 694 new reviews in Run 3, with 1,306 overlapping reviews.

All Google Play repeated-run comparisons showed the same schema across runs.

This confirms that Google Play is technically feasible for a small recurring review ingestion pilot. It can repeatedly return a stable review volume, preserve stable review IDs, support duplicate handling, and capture new reviews over time.

## Overall iOS App Store Result

The iOS App Store public RSS feed was also technically accessible and repeatable, but it was less consistent than Google Play.

Across the three runs, iOS returned useful review volume, but the exact results varied more:

- Run 1 collected 7,000 iOS reviews.
- Run 2 collected 7,000 iOS reviews.
- Run 3 collected 6,800 iOS reviews.

The duplicate counts also varied across runs. This confirms that duplicate handling is necessary for iOS.

The iOS repeated-run comparisons showed that iOS can capture new reviews across app-country pairs. However, iOS depends more heavily on country and page combinations. Some app-country pairs had high overlap across runs, while others captured more new reviews. Some combinations also returned fewer reviews in Run 3, which shows that iOS coverage is less consistent than Google Play.

Compared with Google Play, iOS has stronger limitations around:

- country/page coverage
- empty or incomplete app-country pairs
- duplicate rows
- less consistent collected volume

Based on these results, iOS should remain a secondary source rather than the primary ingestion path.

## Exploratory and Data Quality Findings

The exploratory and data quality checks were consistent with the source-level findings.

The analysis included:

- rating distribution
- review length
- missing fields
- app version coverage
- language detection
- duplicate summaries
- source-level comparison

Main findings:

- Both sources contain a mix of positive and negative reviews.
- Many reviews are either 1-star or 5-star.
- iOS reviews were generally longer than Google Play reviews.
- Google Play core fields were complete.
- Google Play app version and developer reply fields had missing values, which is expected.
- iOS checked fields were mostly complete for collected reviews.
- Language detection showed mostly English reviews, but some reviews were detected as other languages.
- Language detection should be treated as a rough data quality flag because short reviews can be misclassified.

## Final Recommendation

After three runs, Google Play should be treated as the primary source for the recurring app review ingestion pilot.

Google Play is recommended as the primary path because it provides:

- stable repeated collection
- strong and consistent review volume
- stable review IDs
- clean within-run duplicate handling
- schema consistency across runs
- clear new review capture across repeated runs
- usable metadata for downstream analysis

iOS App Store public RSS should remain a secondary source.

iOS is useful for additional coverage, especially across countries, but it should not be the main source because it has more duplicate rows, stronger country/page dependence, and less consistent collection volume.

## Final Module Conclusion

The current module validates that public app review collection is feasible.

The strongest path is to use Google Play as the main recurring ingestion source and keep iOS as a supplementary source. If this work moves into the next stage, the recommended next step is to convert the notebook workflow into reusable scripts and schedule additional recurring runs to continue monitoring freshness, failures, schema stability, and duplicate behavior over time.
"""

synthesized_report_path = os.path.join(
    PROJECT_DIR,
    f"reports/synthesized_module_conclusion_{RUN_DATE}.md"
)

with open(synthesized_report_path, "w") as f:
    f.write(synthesized_report)

print("Saved synthesized module conclusion to:", synthesized_report_path)

Saved synthesized module conclusion to: /content/drive/MyDrive/app_review_source_validation/reports/synthesized_module_conclusion_2026_06_23.md
